In [1]:
"""
MODEL 2 - RAD-DINO + LightGBM Tabular Verifier (Student 3)  |  Kaggle script
------------------------------------------------------------------------
Tests RT-DETR (896px SOTA run) candidate proposals with a LightGBM classifier. Two variants are run side by side:
  V1 "LGBM tab-22"         : spec-faithful, 22-D geometry + detector-context vector only (no pixels).
  V2 "LGBM tab-22 + RADDINO": same 22-D + PCA(32) of the RAD-DINO [CLS||mean-patch] embedding of each RT-DETR crop,
                              so the crops themselves are actually tested.
Protocol (no test leakage):
  A) VAL->TEST : 5-fold (grouped by image) on val_candidate_proposals -> OOF scores pick thresholds;
                 test scores = mean of the 5 fold models on test_candidate_proposals.
  B) TEST 5-FOLD OOF : paper-style pooled out-of-fold benchmark on test proposals.

KAGGLE SETUP: add inputs = (1) your all_final_outputs dataset, (2) RSNA competition data
(rsna-pneumonia-detection-challenge; full images to re-cut crops + GT labels),
(3) optionally a RAD-DINO model dataset, else turn Internet ON (only needed for V2). Accelerator: GPU T4/P100.
"""
# ===================================================================================
# SHARED PLUMBING (identical in both scripts): input discovery, schema detection,
# labels, 22-D tabular features, crop extraction through RAD-DINO, metrics.
# ===================================================================================
import os, sys, json, warnings, random
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import torch
from PIL import Image

# ----------------------------- CONFIG (edit if needed) -----------------------------
INPUT_ROOT     = "/kaggle/input"
WORK           = "/kaggle/working"
SEED           = 42
N_FOLDS        = 5
CROP           = 224          # RAD-DINO input size
MARGIN         = 0.10         # context margin delta
IOU_POS_THRESH = 0.50         # proposal is TP if IoU with a GT box >= this (only used if no label column)
COORD_REF_SIZE = None         # None = auto (1024 for pixel coords). Set 896 if your CSV boxes are in 896px space.
BATCH_IMGS     = 8
NUM_WORKERS    = 2
# Column overrides, e.g. COL_OVR = dict(image="patientId", conf="confidence"). None = auto-detect.
COL_OVR = dict(image=None, x1=None, y1=None, x2=None, y2=None, x=None, y=None, w=None, h=None,
               conf=None, label=None, iou=None)
RADDINO_ID = "microsoft/rad-dino"   # used if no local copy is attached under /kaggle/input

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------- input discovery -----------------------------
FILES, IMGS, CROPS = {}, {}, {}
def scan_inputs():
    for r, _, fs in os.walk(INPUT_ROOT):
        in_crops = "detected_pneumonia_rois" in r.replace("\\", "/")
        for f in fs:
            p = os.path.join(r, f); low = f.lower()
            if low.endswith((".dcm", ".png", ".jpg", ".jpeg")):
                if in_crops: CROPS.setdefault(Path(f).stem, []).append(p)
                else: IMGS.setdefault(Path(f).stem, p)
            else:
                FILES.setdefault(low, p)
    # crops zip that Kaggle did not auto-extract
    z = FILES.get("detected_pneumonia_rois.zip")
    if z and not CROPS:
        import zipfile
        dst = f"{WORK}/rois"; os.makedirs(dst, exist_ok=True)
        zipfile.ZipFile(z).extractall(dst)
        for r, _, fs in os.walk(dst):
            for f in fs:
                if f.lower().endswith((".png", ".jpg", ".jpeg")):
                    CROPS.setdefault(Path(f).stem, []).append(os.path.join(r, f))
    print(f"[scan] csv/other files: {len(FILES)} | full images found: {len(IMGS)} | crop files: {sum(map(len, CROPS.values()))}")
scan_inputs()

def need(name):
    p = FILES.get(name.lower())
    if p is None:
        raise FileNotFoundError(f"'{name}' not found under {INPUT_ROOT}. Attach the all_final_outputs dataset to this notebook.")
    return p

# ----------------------------- schema detection -----------------------------
CAND = dict(
    image=["patientid", "patient_id", "image_id", "imageid", "image", "img_id", "img", "filename", "file_name", "file", "stem", "id"],
    x1=["x1", "xmin", "x_min", "left"], y1=["y1", "ymin", "y_min", "top"],
    x2=["x2", "xmax", "x_max", "right"], y2=["y2", "ymax", "y_max", "bottom"],
    x=["x"], y=["y"], w=["w", "width"], h=["h", "height"],
    conf=["conf", "confidence", "score", "det_conf", "detector_conf", "det_score", "prob"],
    label=["is_tp", "tp", "label", "target", "is_true_positive", "matched", "gt_match", "is_positive", "y_true"],
    iou=["iou", "max_iou", "best_iou", "gt_iou", "iou_gt"])
def pick(df, names):
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n in low: return low[n]
    return None

def norm_id(s):
    s = str(s)
    return Path(s).stem if s.lower().endswith((".dcm", ".png", ".jpg", ".jpeg")) else s

def load_gt():
    for n in ("stage_2_train_labels.csv", "stage_1_train_labels.csv"):
        if n in FILES:
            g = pd.read_csv(FILES[n]); g = g[g["Target"] == 1]
            d = {}
            for pid, s in g.groupby("patientId"):
                d[pid] = np.stack([s.x, s.y, s.x + s.width, s.y + s.height], 1).astype(float)
            return d
    return None

def box_iou(a, b):  # a [n,4], b [m,4] -> [n,m]
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)

def load_split(path, name):
    d = pd.read_csv(path)
    c = {k: (COL_OVR.get(k) or pick(d, CAND[k])) for k in CAND}
    print(f"\n[SCHEMA:{name}] file={Path(path).name} rows={len(d)}\n  columns={list(d.columns)}\n  detected={ {k:v for k,v in c.items() if v} }")
    if c["image"] is None or c["conf"] is None:
        raise ValueError(f"Could not detect image-id / confidence columns in {name}. Set COL_OVR at the top. Columns: {list(d.columns)}")
    if all(c[k] for k in ("x1", "y1", "x2", "y2")):
        b = d[[c["x1"], c["y1"], c["x2"], c["y2"]]].astype(float).values
    elif all(c[k] for k in ("x", "y", "w", "h")):
        b = d[[c["x"], c["y"], c["w"], c["h"]]].astype(float).values; b[:, 2] += b[:, 0]; b[:, 3] += b[:, 1]
    else:
        raise ValueError(f"No box columns detected in {name}. Set COL_OVR (x1,y1,x2,y2 or x,y,w,h).")
    mx = np.nanmax(b)
    ref = 1.0 if mx <= 1.5 else float(COORD_REF_SIZE or 1024)
    print(f"  max box coord={mx:.1f} -> treated as {'normalized' if ref == 1.0 else f'pixels in {int(ref)}px space'}")
    b = np.clip(b / ref, 0, 1)
    out = pd.DataFrame({"img": d[c["image"]].map(norm_id), "nx1": b[:, 0], "ny1": b[:, 1], "nx2": b[:, 2], "ny2": b[:, 3],
                        "conf": d[c["conf"]].astype(float).values})
    out = out[(out.nx2 > out.nx1) & (out.ny2 > out.ny1)].reset_index(drop=True)
    keep = ((b[:, 2] > b[:, 0]) & (b[:, 3] > b[:, 1]))
    y = iou = None
    if c["label"]: y = (d[c["label"]].astype(float).values > 0.5)[keep]
    if c["iou"]:
        iou = d[c["iou"]].astype(float).clip(0, 1).values[keep]
        if y is None: y = iou >= IOU_POS_THRESH
    if y is None or iou is None:
        gt = load_gt()
        if gt is not None:
            iou_c = np.zeros(len(out))
            for im, idx in out.groupby("img").indices.items():
                if im in gt:
                    pb = out.iloc[idx][["nx1", "ny1", "nx2", "ny2"]].values * 1024.0
                    iou_c[idx] = box_iou(pb, gt[im]).max(1)
            if iou is None: iou = iou_c
            if y is None: y = iou_c >= IOU_POS_THRESH
            print("  labels/IoU computed from RSNA ground-truth CSV")
        elif y is None:
            raise ValueError(f"{name}: no label/IoU column and no stage_2_train_labels.csv attached. Attach the RSNA competition data or set COL_OVR['label'].")
    out["y"] = y.astype(bool)
    out["iou_t"] = iou.astype(float) if iou is not None else out["y"].astype(float)
    out.attrs["aux_iou"] = iou is not None
    print(f"  proposals={len(out)} images={out.img.nunique()} TP={int(out.y.sum())} ({out.y.mean():.1%}) FP/img(baseline)={(~out.y).sum() / out.img.nunique():.2f}")
    return out

# ----------------------------- 22-D tabular features (Student-3 spec) -----------------------------
TAB_COLS = ["x1", "y1", "x2", "y2", "cx", "cy", "w", "h", "area", "aspect", "midline", "quadrant",           # 12 geometry
            "conf", "rank", "n_prop", "rel_conf", "conf_gap", "img_mean", "img_std", "max_overlap", "mean_overlap", "cluster_density"]  # 10 context
def tab_features(d):
    f = pd.DataFrame(index=d.index)
    f["x1"], f["y1"], f["x2"], f["y2"] = d.nx1, d.ny1, d.nx2, d.ny2
    f["w"] = d.nx2 - d.nx1; f["h"] = d.ny2 - d.ny1
    f["cx"] = (d.nx1 + d.nx2) / 2; f["cy"] = (d.ny1 + d.ny2) / 2
    f["area"] = f.w * f.h; f["aspect"] = f.w / (f.h + 1e-6); f["midline"] = (f.cx - 0.5).abs()
    f["quadrant"] = (f.cx > 0.5).astype(int) + 2 * (f.cy > 0.5).astype(int)
    g = d.groupby("img")["conf"]
    f["conf"] = d.conf; f["rank"] = g.rank(ascending=False, method="first"); f["n_prop"] = g.transform("size")
    f["rel_conf"] = d.conf / (g.transform("max") + 1e-9); f["conf_gap"] = g.transform("max") - d.conf
    f["img_mean"] = g.transform("mean"); f["img_std"] = g.transform("std").fillna(0)
    mo, me, cd = np.zeros(len(d)), np.zeros(len(d)), np.zeros(len(d))
    for _, idx in d.groupby("img").indices.items():
        if len(idx) < 2: continue
        b = d.iloc[idx][["nx1", "ny1", "nx2", "ny2"]].values
        m = box_iou(b, b); np.fill_diagonal(m, 0)
        mo[idx] = m.max(1); me[idx] = m.sum(1) / (len(idx) - 1); cd[idx] = (m > 0.3).sum(1)
    f["max_overlap"], f["mean_overlap"], f["cluster_density"] = mo, me, cd
    return f[TAB_COLS].astype(np.float32)

# ----------------------------- crops -> RAD-DINO features -----------------------------
MEAN = np.array([0.485, 0.456, 0.406], np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], np.float32)[:, None, None]
def prep(im):
    x = np.asarray(im.convert("L").resize((CROP, CROP), Image.BICUBIC), np.float32) / 255.0
    return torch.from_numpy(((np.stack([x] * 3, 0) - MEAN) / STD).astype(np.float32))

def load_gray(path):
    if path.lower().endswith(".dcm"):
        import pydicom
        ds = pydicom.dcmread(path); a = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1": a = a.max() - a
        a = (a - a.min()) / (a.max() - a.min() + 1e-6) * 255
    else:
        a = np.asarray(Image.open(path).convert("L"), np.float32)
    return a.astype(np.uint8)

def crop_box(a, bb, margin):
    H, W = a.shape; x1, y1, x2, y2 = bb; bw, bh = x2 - x1, y2 - y1
    x1, x2 = x1 - margin * bw, x2 + margin * bw; y1, y2 = y1 - margin * bh, y2 + margin * bh
    x1, y1 = int(max(0, np.floor(x1))), int(max(0, np.floor(y1))); x2, y2 = int(min(W, np.ceil(x2))), int(min(H, np.ceil(y2)))
    if x2 - x1 < 8: x2 = min(W, x1 + 8); x1 = max(0, x2 - 8)
    if y2 - y1 < 8: y2 = min(H, y1 + 8); y1 = max(0, y2 - 8)
    return Image.fromarray(a[y1:y2, x1:x2])

class ProposalDS(torch.utils.data.Dataset):
    def __init__(self, groups, views): self.g, self.views = groups, views
    def __len__(self): return len(self.g)
    def __getitem__(self, i):
        img, rows, boxes = self.g[i]; L, C = [], []; ok = []
        path = IMGS.get(img)
        if path is not None:
            a = load_gray(path); H, W = a.shape
            for r, b in zip(rows, boxes):
                bb = b * np.array([W, H, W, H]); L.append(prep(crop_box(a, bb, 0.0)))
                if "ctx" in self.views: C.append(prep(crop_box(a, bb, MARGIN)))
                ok.append(r)
        elif img in CROPS and len(CROPS[img]) == len(rows):      # fallback: pre-saved RT-DETR crops (rows are conf-desc sorted)
            for r, cp in zip(rows, sorted(CROPS[img])):
                t = prep(Image.open(cp)); L.append(t)
                if "ctx" in self.views: C.append(t)
                ok.append(r)
        z = torch.zeros(0, 3, CROP, CROP)
        return np.array(ok, int), (torch.stack(L) if L else z), (torch.stack(C) if C else z)

def load_backbone():
    from transformers import AutoModel
    src = RADDINO_ID
    for r, _, fs in os.walk(INPUT_ROOT):
        if "config.json" in fs and "rad" in r.lower() and "dino" in r.lower(): src = r; break
    print(f"[backbone] loading RAD-DINO from: {src}")
    m = AutoModel.from_pretrained(src).to(DEVICE).eval()
    return m.half() if DEVICE == "cuda" else m

@torch.no_grad()
def extract_feats(df, tag, views, backbone):
    """Returns loc[N,1536], ctx[N,1536]|None, valid[N]; feature = [CLS || mean(patch tokens)] per view. Cached."""
    cache = f"{WORK}/cache_{tag}_{'_'.join(views)}.npz"
    if os.path.exists(cache):
        z = np.load(cache); print(f"[feats] cache hit {cache}")
        return z["loc"], (z["ctx"] if "ctx" in z.files else None), z["valid"]
    groups = []
    for img, sub in df.groupby("img", sort=False):
        s = sub.sort_values("conf", ascending=False)
        groups.append((img, s.index.values, s[["nx1", "ny1", "nx2", "ny2"]].values))
    dl = torch.utils.data.DataLoader(ProposalDS(groups, views), batch_size=BATCH_IMGS, num_workers=NUM_WORKERS,
                                     collate_fn=lambda b: b)
    N = len(df); loc = np.zeros((N, 1536), np.float16); ctx = np.zeros((N, 1536), np.float16) if "ctx" in views else None
    valid = np.zeros(N, bool)
    def run(x):
        out = []
        for i in range(0, len(x), 128):
            t = x[i:i + 128].to(DEVICE); t = t.half() if DEVICE == "cuda" else t
            h = backbone(pixel_values=t).last_hidden_state
            out.append(torch.cat([h[:, 0], h[:, 1:].mean(1)], 1).float().cpu().numpy())
        return np.concatenate(out)
    for bi, batch in enumerate(dl):
        rows = np.concatenate([b[0] for b in batch]); 
        if len(rows) == 0: continue
        L = torch.cat([b[1] for b in batch]); loc[rows] = run(L).astype(np.float16); valid[rows] = True
        if ctx is not None: ctx[rows] = run(torch.cat([b[2] for b in batch])).astype(np.float16)
        if bi % 50 == 0: print(f"  [{tag}] {bi * BATCH_IMGS}/{len(groups)} images")
    print(f"[feats] {tag}: valid {valid.sum()}/{N} proposals (missing = no full image and no matching crop file)")
    np.savez(cache, loc=loc, valid=valid, **({"ctx": ctx} if ctx is not None else {}))
    return loc, ctx, valid

# ----------------------------- metrics -----------------------------
from sklearn.metrics import roc_auc_score, average_precision_score
def op_point(y, keep, n_img):
    tp = int((keep & y).sum()); fp = int((keep & ~y).sum()); P = int(y.sum())
    return dict(precision=round(tp / max(tp + fp, 1), 4), tp_retention=round(tp / max(P, 1), 4),
                fp_per_img=round(fp / n_img, 4), kept=int(keep.sum()))
def thr_for_retention(y, s, target): return float(np.quantile(s[y], 1 - target))

def report(tag, scores, y_cal, s_cal, y, n_img, det):
    """scores: {name: test score}; s_cal: {name: calibration-set score} used to pick thresholds (no test peeking)."""
    rows = []
    base = op_point(y, np.ones(len(y), bool), n_img)
    rows.append(dict(model="all proposals (no filter)", AUROC=np.nan, AUPRC=np.nan, **{"op": "-"}, **base))
    for name, s in scores.items():
        auroc, auprc = roc_auc_score(y, s), average_precision_score(y, s)
        ops = [("tau=0.50", 0.50)] if name != "detector conf" else []
        for t in (0.90, 0.80): ops.append((f"ret>={int(t * 100)}% (cal-set thr)", thr_for_retention(y_cal, s_cal[name], t)))
        for opn, thr in ops:
            rows.append(dict(model=name, AUROC=round(auroc, 4), AUPRC=round(auprc, 4), op=opn, **op_point(y, s >= thr, n_img)))
    df = pd.DataFrame(rows); print(f"\n=== {tag} ==="); print(df.to_string(index=False)); return df

def fuse(det, p, alpha=0.70): return alpha * det + (1 - alpha) * p

def plot_curves(path, y, scores, title):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve, roc_curve
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    for n, s in scores.items():
        fpr, tpr, _ = roc_curve(y, s); ax[0].plot(fpr, tpr, label=f"{n} ({roc_auc_score(y, s):.3f})")
        pr, rc, _ = precision_recall_curve(y, s); ax[1].plot(rc, pr, label=f"{n} ({average_precision_score(y, s):.3f})")
    ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set(xlabel="FPR", ylabel="TPR", title="ROC"); ax[1].set(xlabel="Recall", ylabel="Precision", title="PR")
    for a in ax: a.legend(fontsize=8); a.grid(alpha=.3)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

from sklearn.model_selection import GroupKFold
def prepare_split(name, fname):
    df = load_split(need(fname), name)
    return df

def attach(df, tag, views, bb):
    loc, ctx, valid = extract_feats(df, tag, views, bb)
    keep = np.where(valid)[0]
    d2 = df.iloc[keep].reset_index(drop=True)
    return d2, loc[keep], (ctx[keep] if ctx is not None else None)

# ======================= MODEL 2 SPECIFIC =======================
import lightgbm as lgb
from sklearn.decomposition import PCA
OUT = f"{WORK}/model2_lightgbm_out"; os.makedirs(OUT, exist_ok=True)
PCA_DIM = 32
LGB_PARAMS = dict(objective="binary", n_estimators=500, learning_rate=0.03, num_leaves=15, min_child_samples=20,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, n_jobs=4)

def cv_lgb(X, y, groups, extra=()):
    oof = np.zeros(len(y)); ev = [np.zeros(len(e)) for e in extra]
    for k, (tr, te) in enumerate(GroupKFold(N_FOLDS).split(X, y, groups)):
        m = lgb.LGBMClassifier(random_state=SEED + k, **LGB_PARAMS).fit(X[tr], y[tr])
        oof[te] = m.predict_proba(X[te])[:, 1]
        for j, e in enumerate(extra): ev[j] += m.predict_proba(e)[:, 1] / N_FOLDS
    return oof, ev

def with_emb(tab, emb, pca):
    return np.hstack([tab, pca.transform(emb.astype(np.float32))]).astype(np.float32)

bb = None
te_raw = prepare_split("TEST", "test_candidate_proposals.csv")
has_val = "val_candidate_proposals.csv" in FILES
need_emb = True
try: bb = load_backbone()
except Exception as e:
    print(f"[warn] RAD-DINO could not be loaded ({e}); running V1 (tabular-only) only"); need_emb = False

def get(raw, tag):
    if need_emb: d, loc, _ = attach(raw, tag, ("loc",), bb)
    else: d, loc = raw, None
    return d, tab_features(d).values, loc
te, te_tab, te_emb = get(te_raw, "test"); y_te = te.y.values; n_te = te.img.nunique()
variants = [("LGBM tab-22", False)] + ([("LGBM tab-22 + RAD-DINO crops", True)] if need_emb else [])
summary = {}

if has_val:
    va, va_tab, va_emb = get(prepare_split("VAL", "val_candidate_proposals.csv"), "val"); y_va = va.y.values
    sc_te = {"detector conf": te.conf.values}; sc_va = {"detector conf": va.conf.values}
    print("\n>>> A) VAL -> TEST")
    for name, use_emb in variants:
        if use_emb:
            pca = PCA(PCA_DIM, random_state=SEED).fit(va_emb.astype(np.float32))        # unsupervised, fit on VAL only
            Xva, Xte = with_emb(va_tab, va_emb, pca), with_emb(te_tab, te_emb, pca)
        else: Xva, Xte = va_tab, te_tab
        oof, (p,) = cv_lgb(Xva, y_va, va.img.values, extra=[Xte])
        sc_te[name], sc_va[name] = p, oof
        sc_te[name + " + det fusion (a=0.7)"], sc_va[name + " + det fusion (a=0.7)"] = fuse(te.conf.values, p), fuse(va.conf.values, oof)
        if not use_emb:
            m = lgb.LGBMClassifier(random_state=SEED, **LGB_PARAMS).fit(Xva, y_va)
            pd.Series(m.booster_.feature_importance("gain"), index=TAB_COLS).sort_values(ascending=False).to_csv(f"{OUT}/feature_importance_gain.csv")
    rA = report("A) Train on VAL, test on TEST (thresholds from VAL OOF)", sc_te, y_va, sc_va, y_te, n_te, te.conf.values)
    rA.to_csv(f"{OUT}/metrics_val_to_test.csv", index=False)
    plot_curves(f"{OUT}/curves_val_to_test.png", y_te, sc_te, "RAD-DINO + LightGBM: VAL->TEST")
    pd.DataFrame({"img": te.img, "nx1": te.nx1, "ny1": te.ny1, "nx2": te.nx2, "ny2": te.ny2, "det_conf": te.conf, "y": y_te,
                  **{"p_" + k.replace(" ", "_").replace("+", "plus"): v for k, v in sc_te.items() if "fusion" not in k and k != "detector conf"}}
                 ).to_csv(f"{OUT}/test_predictions_lgbm.csv", index=False)
    summary["val_to_test"] = rA.to_dict("records")
else:
    print("val_candidate_proposals.csv not found -> skipping mode A")

print("\n>>> B) TEST 5-fold pooled OOF (paper-style)")
sc = {"detector conf": te.conf.values}
for name, use_emb in variants:
    X = with_emb(te_tab, te_emb, PCA(PCA_DIM, random_state=SEED).fit(te_emb.astype(np.float32))) if use_emb else te_tab
    oof, _ = cv_lgb(X, y_te, te.img.values)
    sc[name] = oof; sc[name + " + det fusion (a=0.7)"] = fuse(te.conf.values, oof)
rB = report("B) TEST pooled 5-fold OOF (thresholds from the same OOF pool)", sc, y_te, sc, y_te, n_te, te.conf.values)
rB.to_csv(f"{OUT}/metrics_test_oof.csv", index=False)
plot_curves(f"{OUT}/curves_test_oof.png", y_te, sc, "RAD-DINO + LightGBM: TEST 5-fold OOF")
summary["test_oof"] = rB.to_dict("records")
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2, default=float)
print(f"\nAll outputs in {OUT}")

[scan] csv/other files: 11 | full images found: 29685 | crop files: 7575

[SCHEMA:TEST] file=test_candidate_proposals.csv rows=25610
  columns=['patientId', 'x1', 'y1', 'x2', 'y2', 'conf', 'iou_with_gt', 'is_tp']
  detected={'image': 'patientId', 'x1': 'x1', 'y1': 'y1', 'x2': 'x2', 'y2': 'y2', 'conf': 'conf', 'label': 'is_tp'}
  max box coord=1023.9 -> treated as pixels in 1024px space
  labels/IoU computed from RSNA ground-truth CSV
  proposals=25610 images=2993 TP=1445 (5.6%) FP/img(baseline)=8.07
[backbone] loading RAD-DINO from: microsoft/rad-dino


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  [test] 0/2993 images
  [test] 400/2993 images
  [test] 800/2993 images
  [test] 1200/2993 images
  [test] 1600/2993 images
  [test] 2000/2993 images
  [test] 2400/2993 images
  [test] 2800/2993 images
[feats] test: valid 25610/25610 proposals (missing = no full image and no matching crop file)

[SCHEMA:VAL] file=val_candidate_proposals.csv rows=13054
  columns=['patientId', 'x1', 'y1', 'x2', 'y2', 'conf', 'iou_with_gt', 'is_tp']
  detected={'image': 'patientId', 'x1': 'x1', 'y1': 'y1', 'x2': 'x2', 'y2': 'y2', 'conf': 'conf', 'label': 'is_tp'}
  max box coord=1023.8 -> treated as pixels in 1024px space
  labels/IoU computed from RSNA ground-truth CSV
  proposals=13054 images=1495 TP=733 (5.6%) FP/img(baseline)=8.24
  [val] 0/1495 images
  [val] 400/1495 images
  [val] 800/1495 images
  [val] 1200/1495 images
[feats] val: valid 13054/13054 proposals (missing = no full image and no matching crop file)

>>> A) VAL -> TEST

=== A) Train on VAL, test on TEST (thresholds from VAL OOF) ===
 